In [95]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
import pandas as pd
from pandas import DataFrame as DF
from sklearn.preprocessing import MinMaxScaler

# Load the CSV file into a DataFrame
df = pd.read_csv('lexical.csv')

# Exclude the first column
df = df.iloc[:, 1:-1]
X= df
for column in X.columns:
    if type(X[column].iloc[0]) != type(X[X.columns[-1]].iloc[0]):
        X[column] = X[column].astype(float)
yList= []
for i in range(df.shape[0]):
    yList.append(np.random.randint(0, 2))

y = DF(yList)
for column in y.columns:
    if type(y[column].iloc[0]) != type(y[y.columns[-1]].iloc[0]):
        y[column] = y[column].astype(float)


print(X.shape)
print(y.shape)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

for column in ["len_url","len_component", "count_letters_component", "ratio_digits_component_url", "ratio_letters_component_url", "character_continuity_rate_url"]:
    scaler = MinMaxScaler()
    scaler.fit(X_train[[column]])
    X_train[[column]] = scaler.transform(X_train[[column]])

X_train[["use_https"]] = X_train[["use_https"]].astype(float)

# Create and train the Random Forest classifier
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred_rf = rf_model.predict(X_test)

# Evaluate the Random Forest model
accuracy_rf = accuracy_score(y_test, y_pred_rf)
conf_matrix_rf = confusion_matrix(y_test, y_pred_rf)

# Display results
print(f"Random Forest Accuracy: {accuracy_rf}")

(74623, 15)
(74623, 1)


C:\Users\James\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:1474: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


Random Forest Accuracy: 0.4970854271356784


In [91]:
print(rf_model.predict(np.array(X_test.iloc[2]).reshape(1,-1)))
X_test.shape

[0]


C:\Users\James\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


(14925, 15)

In [96]:
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
from sklearn.ensemble import RandomForestClassifier  # Import the appropriate model

# Assuming 'model' is your trained Random Forest model
initial_type = [('float_input', FloatTensorType([None, 15]))]
onnx_model = convert_sklearn(rf_model, initial_types=initial_type)

with open("random_forest_model.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())

"""import joblib

joblib.dump(rf_model, "model.joblib")"""

'import joblib\n\njoblib.dump(rf_model, "model.joblib")'

In [97]:


for column in X.columns:
    print(type(X[column].iloc[0]))
    '''if type(X[column].iloc[0]) != type(X[X.columns[-1]].iloc[0]):
        X[column] = X[column].astype(float)
        print("stillTrue")'''

<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>


In [27]:
import onnxruntime

ort_session = onnxruntime.InferenceSession("LogisticRegression.onnx", providers=["CPUExecutionProvider"])

def to_numpy(tensor):
    return tensor.detach().cpu().numpy() if tensor.requires_grad else tensor.cpu().numpy()

# compute ONNX Runtime output prediction
tstList = X_test.values.tolist()
ort_inputs = {ort_session.get_inputs()[0].name: X_test}
ort_outs = ort_session.run(None, ort_inputs)

# compare ONNX Runtime and PyTorch results
np.testing.assert_allclose(output, ort_outs[0], rtol=1e-03, atol=1e-05)

print("Exported model has been tested with ONNXRuntime, and the result looks good!")

RuntimeError: Input must be a list of dictionaries or a single numpy array for input 'float_input'.

In [25]:
for column in ["len_url","len_component", "count_letters_component", "ratio_digits_component_url", "ratio_letters_component_url", "character_continuity_rate_url"]:
    scaler = MinMaxScaler()
    scaler.fit(X_test[[column]])